In [1]:
# Imports
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from diffrax import *
from controls import *

In [11]:
# (1) Dynamics of stoch vol. We want to solve this one without any correlation/lead-lag. Therefore we just use an Ito solver.

gamma1 = 1
gamma2 = 1 # note this is not Lipschitz and could explode...
sigma1 = 1.5
sigma2 = 1.5
alpha = 1.5
beta = 1.5

t0, t1 = 0, 1
epsilon = 0.01
lag = epsilon # = 1.2*epsilon was not really necessary
solver_epsilon = 0.1*epsilon # = 0.01*epsilon is really necessary in these examples
times = jnp.arange(t0, t1, epsilon)
num_samples = 1
dim_bm = 2

y0 = jnp.array([1, 1])
drift_jones = lambda t, y, args: jnp.array([0, alpha + beta*y[1]])
diffusion_jones = lambda t, y, args: jnp.array([[jnp.sqrt(y[1])*y[0], 0], [sigma1*jnp.power(y[1], gamma1), sigma2*jnp.power(y[1], gamma2)]])
saveat = SaveAt(ts = times)

key=jr.PRNGKey(512)
split_key = jax.random.split(key, num_samples)

solutions = vmap_batch_solve(split_key, epsilon, dim_bm, drift_jones, diffusion_jones, y0, Midpoint(), t0, t1, solver_epsilon, saveat)